# Choosing where to put the dye

Before any FRET experiment there is a question that has nothing to do with photons:
**which residue should carry the label, and which pair should be measured?**

The Labelizer score (Gebhardt *et al.*, *Nat. Commun.* **16**, 3305, 2025) answers it
by combining four per-residue quantities — how conserved a position is, how exposed
it is, what secondary structure it sits in, and how much the residue already resembles
a cysteine — into one number, and then scoring pairs of the good sites by how
informative a FRET measurement between them would be.

`IMP.bff` carries a 1:1 native port of that model. This page walks it end to end and,
along the way, shows what happened when it was checked against the reference
implementation's own published output.

Two things to know before reading any number:

- **A score is a likelihood ratio, not a probability.** Each fitted table holds
  $P(\text{labelable} \mid s) / P(\text{labelable})$, so 1 is the base rate and the
  combined score is unbounded above.
- **The published arithmetic is the default, defects included**, because that is what
  the paper's numbers were computed with. `LL_MODEL_CORRECTED` selects the arithmetic
  the reference documents instead, and a written container always records which was
  used.


## The structure

Mouse BID (PDB 1DDB, model 39, chain A) — the worked example the reference ships,
which is what makes the comparison below possible at all.

Conservation is **imported, never computed**: an alignment and a rate estimate are a
different program. `ll_read_consurf` takes a ConSurf `.grades` table or a PDB carrying
the normalised grade in its B-factor column.


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

import IMP.bff as bff

data = os.path.join('..', '..', '..', 'test', 'input', 'labelizer')
pdb = os.path.join(data, '1DDB-39.pdb')
conservation = os.path.join(data, '1DDB-39_cs.pdb')

structure = bff.ll_read_structure(pdb)
print(f'{len(structure.residues)} residues, {len(structure.vdw)} atoms, '
      f'chains {sorted({r.chain for r in structure.residues})}')


## The structural features

These are the quantities the model scores. Each is a kernel of its own, useful
outside the score.

`ll_dssp` is the interesting one. The reference shells out to the DSSP binary and
raises `RuntimeError("Unknown platform")` on macOS, so the secondary-structure term
simply could not be evaluated. This is a native Kabsch–Sander implementation.


In [ ]:
secondary = bff.ll_dssp(structure)
depth = np.asarray(bff.ll_residue_depth(structure, 1.4, 590))
rsa = np.asarray(bff.ll_relative_solvent_accessibility(
    structure, bff.LL_MAXASA_WILKE, 1.4, 590))
hse = np.asarray(bff.ll_half_sphere_exposure(structure, 13.0)).reshape(-1, 2)

print(secondary[:90])
print({c: secondary.count(c) for c in sorted(set(secondary))})
print(f'depth {depth.min():.2f}-{depth.max():.2f} A, '
      f'RSA {rsa.min():.2f}-{rsa.max():.2f}')


BID is all-helical, and the assignment says so: 96 `H`, no strands at all.

### Does it agree with the real DSSP?

It can be checked exactly. The reference's `ss` score is a lookup on the DSSP letter,
so inverting its published score recovers the letter its binary assigned.


In [ ]:
table = bff.ll_load_table('C_SS1_SS')
by_value = {round(v, 9): k for k, v in dict(table.by_key).items()}

import csv
with open(os.path.join(data, '1DDB-39_ss.csv')) as fh:
    reference = {r[0]: float(r[1]) for r in list(csv.reader(fh))[1:]}

disagree = [
    (bff.ll_residue_key(r.chain, r.seq_id),
     by_value[round(reference[bff.ll_residue_key(r.chain, r.seq_id)], 9)],
     secondary[i])
    for i, r in enumerate(structure.residues)
    if by_value[round(reference[bff.ll_residue_key(r.chain, r.seq_id)], 9)]
       != secondary[i]
]
print(f'{len(structure.residues) - len(disagree)}/{len(structure.residues)}'
      f' residues agree with the reference DSSP')
print('disagreements:', disagree)


**All 195.** Two corrections were needed to get there, and both are real:

1. DSSP marks the residues *between* an n-turn's hydrogen-bonded pair — `k+1 … k+n-1`,
   not the donor `k`. Including `k` over-assigned `T`.
2. **DSSP 3.0 reversed the π/α precedence** (Touw 2015): where a 4-turn and a 5-turn
   overlap, the π-helix now wins. That accounted for the last four residues, and it
   dates the reference's own output to DSSP ≥ 3.

The solvent-exposure term cannot be exact in the same way — its published form is MSMS
residue depth, and the MSMS binaries the reference ships are 32-bit ppc/i386 Mach-O
that do not execute. The native surface agrees with **no bias** (mean +0.015 Å,
r = 0.976, 92 % within one table bin). See `okf/validation/labelizer_ab.md`.


## The score

The published model weights conservation, solvent exposure, cysteine resemblance and
secondary structure at 1, and switches tryptophan proximity and charge environment
**off** — they are weight 0 in the paper, not merely down-weighted.


In [ ]:
model = bff.ll_model_paper()
print(', '.join(f'{p.tag}(w={p.weight}, {p.table or "hard-coded"})' for p in model))

scores = bff.ll_score_structure(pdb, model, bff.LlOptions(), conservation)
print(f'{len(scores)} rows: one per (position, score_type)')

combined = {bff.ll_residue_key(r.asym_id, r.seq_id): r.value
            for r in scores
            if r.score_type == 'combined' and r.status == 'scored'}
for key, value in sorted(combined.items(), key=lambda kv: -kv[1])[:8]:
    print(f'  {key:<6} {value:.4f}')


Serine, glutamine and threonine on exposed loops — which is what the
cysteine-resemblance table encodes (S 2.19, Q 1.95, T 1.58 against W 0.21).

The rows are **tidy**: one per (position, score type), which is the shape the container
stores and the shape a dataframe wants. A position that was not scored carries a
`status` and **no value** — never a sentinel.


In [ ]:
by_type = {}
for r in scores:
    by_type.setdefault(r.score_type, []).append(r)
for score_type in sorted(by_type):
    rows = by_type[score_type]
    n = sum(1 for r in rows if r.status == 'scored')
    print(f'  {score_type:<22} {n:>4} scored, {len(rows) - n:>3} not')


## Which pair to measure

A pair is informative when both sites are labelable **and** the dye–dye distance sits
where FRET responds to it, which is near $R_0$. The score is
$jls \cdot (1 - 2|E - 0.5|)$, maximal at $R = R_0$.

Placing the dye is a cost ladder. The analytic *alpha cone* is the reference's cheap
estimate; a real accessible volume is the expensive truth. So every pair is screened
with the cone and the best `n_refine` are rebuilt — which is the reference's design,
and it turns out to matter a great deal.


In [ ]:
options = bff.LlFretOptions()
options.forster_radius = 52.0
options.n_refine = 5

pairs = list(bff.ll_pair_scores(pdb, combined, options))
above = sum(1 for v in combined.values()
            if v >= options.label_score_threshold)
print(f'{len(pairs)} pairs from the {above} sites at or above the '
      f'label-score threshold of {options.label_score_threshold}')

refined = [p for p in pairs if p.probe_model == bff.PROBE_MODEL_ACCESSIBLE_VOLUME]
print('\nthe five the cone ranked best, after rebuilding their dye clouds:')
for p in refined:
    print(f'  {p.asym_id_1}{p.seq_id_1}-{p.asym_id_2}{p.seq_id_2:<5} '
          f'{p.value:.4f}  d={p.distance:.1f} A')
print('\nthe best pairs now:')
for p in pairs[:5]:
    print(f'  {p.asym_id_1}{p.seq_id_1}-{p.asym_id_2}{p.seq_id_2:<5} '
          f'{p.value:.4f}  d={p.distance:.1f} A')


The refined pairs have **fallen out of the top five**. The cone places the dye 2–5 Å
too far out, and because the pair score peaks sharply at $R = R_0$ that is a ~30 %
score error. This is why `n_refine` should not be left at zero for real work — the
cheap screen is for ranking candidates, not for choosing between them.


## One file out

The reference writes six CSVs, four PDBs carrying a dimensionless score in the
B-factor column, a heat-map JSON and a zip. This writes **one container**: the
structure verbatim, the scores as tables whose columns are named by MMFDB dictionary
items, the complete settings, and provenance edges tying them together.


In [ ]:
out = '1DDB-39.mmfdb.pto'
settings = bff.ll_settings_json(model, bff.LlOptions(), options, conservation)
bff.ll_write_pto(out, pdb, scores, pairs[:200], settings)
print(f'{out}  {os.path.getsize(out)} bytes')

# The structure comes back byte for byte, verified against its checksum.
digest = bff.ll_extract_pto_structure(out, 'recovered.pdb')
print('recovered, sha256 verified:', digest[:16], '...')
print('identical:', open(pdb, 'rb').read() == open('recovered.pdb', 'rb').read())

back = bff.ll_read_pto_scores(out)
print(f'{len(back)} score rows read back')


### A caveat the reference carries, and this port inherits

The shipped example passes its conservation PDB as **both input and output**, so each
run feeds the previous run's scores back in as grades. The lookup has a two-cycle
(1.63 → 2.3553…, 2.36 → 1.6322…, each rounding back to the other's grade), so the
example's conservation term is stuck on exactly two values across 195 residues where
the table has ten bins.

The lookup itself reproduces both to sixteen digits — the machinery is exact; the
input is not. It is a good argument for a container that records what produced a
number, and against writing an output to the path an input was read from.


In [ ]:
t = bff.ll_load_table('N_CS2_Score')
print(bff.ll_lookup(t, 1.63), bff.ll_lookup(t, 2.36))
print('distinct conservation values in the example:',
      len({r.value for r in scores if r.score_type == 'conservation'}))


## The picture


In [ ]:
seq = [r.seq_id for r in structure.residues]
values = [combined.get(bff.ll_residue_key(r.chain, r.seq_id), np.nan)
          for r in structure.residues]

fig, (ax, bx) = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True,
                             gridspec_kw={'height_ratios': [2, 1]})
ax.plot(seq, values, lw=1.0, color='#3b6ea5')
ax.axhline(1.0, color='0.6', lw=0.8, ls='--')
ax.axhline(0.5, color='#b5651d', lw=0.8, ls=':')
ax.set_ylabel('combined label score')
ax.set_title('Labelizer score along mouse BID (1DDB model 39, chain A)')
ax.text(0.99, 0.06, 'dashed: base rate   dotted: pairing threshold',
        transform=ax.transAxes, ha='right', fontsize=8, color='0.4')

bx.plot(seq, depth, lw=1.0, color='#5a5a5a', label='residue depth (A)')
bx.plot(seq, rsa * 4.0, lw=1.0, color='#7aa457', label='RSA (x4)')
bx.set_xlabel('residue')
bx.set_ylabel('exposure')
bx.legend(fontsize=8, loc='upper right')
fig.tight_layout()
